In [11]:
import warnings
warnings.filterwarnings('ignore')

In [19]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ==================== 第一步：读取原始数据 ====================
file_path = "消费者画像数据.xlsx"

population_df = pd.read_excel(file_path, sheet_name='日均客流&人口')
phone_df = pd.read_excel(file_path, sheet_name='手机品牌TOP客流画像占比')
age_df = pd.read_excel(file_path, sheet_name='年龄占比')
gender_df = pd.read_excel(file_path, sheet_name='性别占比')
asset_df = pd.read_excel(file_path, sheet_name='资产等级占比 ')

# ==================== 第二步：只保留需要的列 ====================
phone_columns = ['赢商项目ID'] + ['APPLE', 'HUAWEI', 'SAMSUNG']
phone_df = phone_df[phone_columns]

age_columns = ['赢商项目ID'] + ['19-24', '25-29']
age_df = age_df[age_columns]

gender_columns = ['赢商项目ID', '女性占比']
gender_df = gender_df[gender_columns]

asset_columns = ['赢商项目ID'] + ['超级富豪', '富豪', '中产']
asset_df = asset_df[asset_columns]

# ==================== 第三步：合并所有sheet ====================
df = population_df.merge(phone_df, on='赢商项目ID', how='inner')
df = df.merge(age_df, on='赢商项目ID', how='inner')
df = df.merge(gender_df, on='赢商项目ID', how='inner')
df = df.merge(asset_df, on='赢商项目ID', how='inner')

# ==================== 第四步：计算特征指标 ====================
df['年轻占比'] = df['19-24'] + df['25-29']
df['高消费力_资产'] = df['超级富豪'] + df['富豪'] + df['中产']
df['高消费力_手机'] = df['APPLE'] + df['HUAWEI'] + df['SAMSUNG']
df['高消费力'] = (df['高消费力_资产'] + df['高消费力_手机']) / 2

province_score = {
    '上海市': 4, '北京市': 4, '广东省': 4,
    '江苏省': 3, '浙江省': 3, '四川省': 3, '湖北省': 3, '湖南省': 3,
    '河南省': 3, '安徽省': 3, '福建省': 3, '陕西省': 3, '重庆市': 3,
    '天津市': 3, '山东省': 3, '辽宁省': 3,
    '河北省': 2, '江西省': 2, '广西壮族自治区': 2, '云南省': 2,
    '贵州省': 2, '山西省': 2, '吉林省': 2, '黑龙江省': 2,
}
df['省份分数'] = df['省份'].map(province_score)

# ==================== 第五步：读取快闪店文件，获取映射关系 ====================
flash_df = pd.read_excel("快闪店销售详情更新_unpivot.xlsx")
# 去重，保留每个店铺名称对应的李宁商场名称
flash_mapping = flash_df[['店铺名称', '李宁商场名称']].drop_duplicates()

# ==================== 第六步：创建特征表 ====================
all_features_df = df[['李宁商场名称', '年轻占比', '女性占比', '高消费力', '3公里工作人口', '省份分数']].copy()

# ==================== 第七步：用李宁商场名称匹配，获取特征数据 ====================
# 获取需要匹配的李宁商场名称列表
liNing_store_names = flash_mapping['李宁商场名称'].unique().tolist()

# 匹配特征数据
matched_features = all_features_df[all_features_df['李宁商场名称'].isin(liNing_store_names)].copy()


# ==================== 第八步：K-Means聚类 ====================
if len(matched_features) > 0:
    cluster_features = ['年轻占比', '女性占比', '高消费力', '3公里工作人口', '省份分数']
    
    # 标准化
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(matched_features[cluster_features])
    
    # K=3聚类
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    matched_features['客群类型'] = kmeans.fit_predict(X_scaled)
    
    # 命名客群
    name_mapping = {
        0: "一线城市标杆店",
        1: "女性高消潜力店",
        2: "年轻潮流主力店"
    }
    matched_features['客群类型名称'] = matched_features['客群类型'].map(name_mapping)
    
    # ==================== 第九步：将聚类结果（含特征值）映射回店铺名称 ====================
    # 创建李宁商场名称 -> 特征值+客群类型的映射
    feature_mapping = matched_features[['李宁商场名称', '年轻占比', '女性占比', '高消费力', '3公里工作人口', '省份分数', '客群类型名称']].drop_duplicates()
    
    # 合并到原始快闪店映射表
    result_df = flash_mapping.merge(feature_mapping, on='李宁商场名称', how='left')
    
    # ==================== 第十步：输出结果（按店铺名称，包含特征值） ====================
    output_columns = ['店铺名称', '年轻占比', '女性占比', '高消费力', '3公里工作人口', '省份分数', '客群类型名称']
    print(result_df[output_columns].to_string(index=False))
    
    # 保存结果
    result_df[output_columns].to_excel("快闪店聚类结果.xlsx", index=False)
    print("\n✅ 聚类结果已保存至：快闪店聚类结果.xlsx")
    

         店铺名称   年轻占比   女性占比    高消费力  3公里工作人口  省份分数  客群类型名称
     北京荟聚L1中庭 0.5178 0.4956 0.43340   118600   4.0 一线城市标杆店
        北京三里屯 0.5589 0.4181 0.45510   709493   4.0 年轻潮流主力店
      北京王府井大街 0.5270 0.4722 0.43695   528514   4.0 年轻潮流主力店
    大连柏威年L1西厅 0.5250 0.5204 0.46560   336122   3.0 一线城市标杆店
  哈尔滨哈西万达L1中庭 0.6121 0.5283 0.44365   249564   2.0 女性高消潜力店
杭州西溪印象城A座1F中庭 0.5305 0.4787 0.44810   101314   3.0 一线城市标杆店
  昆明顺城购物中心外广场 0.5975 0.5412 0.41565   546750   2.0 女性高消潜力店
     南京新街口步行街 0.5524 0.4506 0.43920   630993   3.0 年轻潮流主力店
  上海新天地时尚I期中庭 0.5458 0.4436 0.44180   892548   4.0 年轻潮流主力店
    沈阳铁西万象汇1F 0.5728 0.5340 0.43570   425467   3.0 女性高消潜力店
  天津南开大悦城麦田广场 0.5505 0.5034 0.46445   480614   3.0 一线城市标杆店
 长春欧亚卖场10号门中庭 0.5067 0.6152 0.49385   198315   2.0 一线城市标杆店
    长沙万象城L1中庭 0.6369 0.5204 0.45500   299350   3.0 女性高消潜力店
 郑州正弘城3号门1F中庭 0.6079 0.4989 0.47090   514988   3.0 女性高消潜力店
        重庆解放碑 0.5750 0.4892 0.43155   417681   3.0 女性高消潜力店

✅ 聚类结果已保存至：快闪店聚类结果.xlsx


In [24]:
# ==================== 生成所有商业体聚类结果 ====================

# 使用前面已经合并好的 df（所有商业体数据）
# 选择聚类特征
cluster_features = ['年轻占比', '女性占比', '高消费力', '3公里工作人口', '省份分数']

# 删除空值
all_malls_df = df[['李宁商场名称', '城市', '省份', '赢商项目ID'] + cluster_features].copy()
all_malls_df = all_malls_df.dropna(subset=cluster_features)

# 标准化
scaler_all = StandardScaler()
X_scaled_all = scaler_all.fit_transform(all_malls_df[cluster_features])

# K=3聚类
kmeans_all = KMeans(n_clusters=3, random_state=42, n_init=10)
all_malls_df['客群类型'] = kmeans_all.fit_predict(X_scaled_all)

# 命名客群
name_mapping = {
    0: "一线城市标杆店",
    1: "女性高消潜力店",
    2: "年轻潮流主力店"
}
all_malls_df['客群类型名称'] = all_malls_df['客群类型'].map(name_mapping)

# 添加省份Tier
province_tier = {
    '上海市': '一线', '北京市': '一线', '广东省': '一线',
    '江苏省': '强二线', '浙江省': '强二线', '四川省': '强二线', '湖北省': '强二线', 
    '湖南省': '强二线', '河南省': '强二线', '安徽省': '强二线', '福建省': '强二线', 
    '陕西省': '强二线', '重庆市': '强二线', '天津市': '强二线', '山东省': '强二线', 
    '辽宁省': '强二线',
    '河北省': '二线', '江西省': '二线', '广西壮族自治区': '二线', '云南省': '二线',
    '贵州省': '二线', '山西省': '二线', '吉林省': '二线', '黑龙江省': '二线',
}
all_malls_df['省份Tier'] = all_malls_df['省份'].map(province_tier)

# 保存结果
output_columns = ['李宁商场名称', '城市', '省份', '省份Tier', '赢商项目ID', 
                  '年轻占比', '女性占比', '高消费力', '3公里工作人口', '省份分数', 
                  '客群类型', '客群类型名称']
all_malls_df[output_columns].to_excel("所有商业体聚类结果.xlsx", index=False)

# 输出统计
print("\n===== 所有商业体各客群类型特征均值 =====")
print(all_malls_df.groupby('客群类型名称')[cluster_features].mean().round(2))


===== 所有商业体各客群类型特征均值 =====
         年轻占比  女性占比  高消费力    3公里工作人口  省份分数
客群类型名称                                    
一线城市标杆店  0.51  0.46  0.35  120493.42  3.24
女性高消潜力店  0.55  0.42  0.41  494449.61  3.74
年轻潮流主力店  0.57  0.49  0.43  271908.67  2.71


In [30]:
# ==================== 计算15个快闪店的系列销售数量占比（剔除推广类） ====================

# ==================== 1. 读取数据 ====================
file_path = "快闪店销售详情更新_unpivot.xlsx"
sales_detail = pd.read_excel(file_path, sheet_name='快闪销售数据')

# ==================== 2. 数据清洗 ====================
# 标准化系列名称
series_mapping = {
    '李宁荣耀金标': '金标',
    '李宁荣耀': '荣耀',
    '国家队': '国家队',
    '其他系列': '其他'
}
sales_detail['系列'] = sales_detail['系列'].map(series_mapping)

# ==================== 3. 剔除推广类（赠品） ====================
sales_detail_filtered = sales_detail[sales_detail['品类'] != '推广类'].copy()
sales_detail_filtered = sales_detail_filtered[pd.notna(sales_detail_filtered['销售数量'])]
sales_detail_filtered = sales_detail_filtered[sales_detail_filtered['销售数量'] > 0]

# ==================== 4. 按店铺+系列汇总销售数量 ====================
summary = sales_detail_filtered.groupby(['店铺名称', '系列'])['销售数量'].sum().reset_index()

# ==================== 5. 计算每个店铺的总销售数量 ====================
total_by_store = summary.groupby('店铺名称')['销售数量'].sum().reset_index()
total_by_store.columns = ['店铺名称', '总销售数量']

# ==================== 6. 计算每个系列的占比 ====================
summary = summary.merge(total_by_store, on='店铺名称')
summary['销售占比'] = summary['销售数量'] / summary['总销售数量']

# ==================== 7. 透视成宽表格式 ====================
series_order = ['金标', '荣耀', '国家队', '其他']
category_ratio_wide = summary.pivot_table(
    index='店铺名称',
    columns='系列',
    values='销售占比',
    fill_value=0
).reset_index()

for s in series_order:
    if s not in category_ratio_wide.columns:
        category_ratio_wide[s] = 0

category_ratio_wide = category_ratio_wide.rename(columns={
    '金标': '金标Proportion',
    '荣耀': '荣耀Proportion',
    '国家队': '国家队Proportion',
    '其他': '其他Proportion'
})

# 归一化
ratio_cols = ['金标Proportion', '荣耀Proportion', '国家队Proportion', '其他Proportion']
row_sum = category_ratio_wide[ratio_cols].sum(axis=1)
for col in ratio_cols:
    category_ratio_wide[col] = category_ratio_wide[col] / row_sum

# ==================== 8. 合并商场特征数据 ====================
df_clean = pd.read_excel("快闪店聚类结果.xlsx")
df_clean = df_clean.rename(columns={'客群类型名称': '客群类型名称'})

final_df = category_ratio_wide.merge(
    df_clean[['店铺名称', '客群类型名称', '年轻占比', '女性占比', '高消费力', '3公里工作人口', '省份分数']],
    on='店铺名称',
    how='inner'
)

# ==================== 9. 保存结果 ====================
output_path = '15个快闪店_系列销售占比_已计算_剔除推广类.xlsx'
final_df.to_excel(output_path, index=False)

# ==================== 10. 输出各客群类型的平均系列占比 ====================
type_summary = final_df.groupby('客群类型名称')[['金标Proportion', '荣耀Proportion', '国家队Proportion', '其他Proportion']].mean().round(4)


# ==================== 改进版模型 ====================
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')


# ==================== 1. 准备训练数据 ====================
train_features = ['年轻占比', '女性占比', '高消费力', '3公里工作人口', '省份分数']
target_cols = ['金标Proportion', '荣耀Proportion', '国家队Proportion', '其他Proportion']

X_train_raw = final_df[train_features].values
y_train = final_df[target_cols].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)

# ==================== 2. 训练随机森林模型 ====================
models = {}
for i, col in enumerate(target_cols):
    rf = RandomForestRegressor(n_estimators=100, random_state=42, min_samples_split=2)
    rf.fit(X_train, y_train[:, i])
    models[col] = rf

# ==================== 3. 预测所有商业体 ====================
df_all_malls = pd.read_excel('所有商业体聚类结果.xlsx')
df_predict_raw = df_all_malls[train_features + ['李宁商场名称', '城市']].copy()
df_predict = df_predict_raw.dropna(subset=train_features)
X_predict = scaler.transform(df_predict[train_features])

for col in target_cols:
    df_predict[col] = models[col].predict(X_predict)

# 归一化
row_sum = df_predict[target_cols].sum(axis=1)
for col in target_cols:
    df_predict[col] = df_predict[col] / row_sum

# ==================== 4. 保存所有商业体预测结果 ====================
output_path = '所有商业体_各系列占比预测.xlsx'
with pd.ExcelWriter(output_path) as writer:
    df_predict.to_excel(writer, sheet_name='所有商业体预测', index=False)
    type_summary.to_excel(writer, sheet_name='各客群类型平均占比')

# ==================== 5. 一线和新一线城市高潜力商业体TOP20 ====================
tier1_cities = ['上海市', '北京市', '深圳市', '广州市']

new_tier1_cities = ['成都市', '杭州市', '重庆市', '武汉市', '苏州市', '西安市', '南京市', 
                     '长沙市', '郑州市', '天津市', '合肥市', '青岛市', '东莞市', '宁波市']

# 一线城市金标TOP20
df_tier1 = df_predict[df_predict['城市'].isin(tier1_cities)].copy()
top_gold_tier1 = df_tier1.nlargest(20, '金标Proportion')[
    ['李宁商场名称', '城市', '金标Proportion', '荣耀Proportion', '国家队Proportion', '其他Proportion']
].copy()
top_gold_tier1 = top_gold_tier1.reset_index(drop=True)

# 新一线城市金标TOP20
df_new_tier1 = df_predict[df_predict['城市'].isin(new_tier1_cities)].copy()
top_gold_new_tier1 = df_new_tier1.nlargest(20, '金标Proportion')[
    ['李宁商场名称', '城市', '金标Proportion', '荣耀Proportion', '国家队Proportion', '其他Proportion']
].copy()
top_gold_new_tier1 = top_gold_new_tier1.reset_index(drop=True)

# 一线城市荣耀TOP20
top_glory_tier1 = df_tier1.nlargest(20, '荣耀Proportion')[
    ['李宁商场名称', '城市', '金标Proportion', '荣耀Proportion', '国家队Proportion', '其他Proportion']
].copy()
top_glory_tier1 = top_glory_tier1.reset_index(drop=True)

# 新一线城市荣耀TOP20
top_glory_new_tier1 = df_new_tier1.nlargest(20, '荣耀Proportion')[
    ['李宁商场名称', '城市', '金标Proportion', '荣耀Proportion', '国家队Proportion', '其他Proportion']
].copy()
top_glory_new_tier1 = top_glory_new_tier1.reset_index(drop=True)

# 一线城市国家队TOP20
top_national_tier1 = df_tier1.nlargest(20, '国家队Proportion')[
    ['李宁商场名称', '城市', '金标Proportion', '荣耀Proportion', '国家队Proportion', '其他Proportion']
].copy()
top_national_tier1 = top_national_tier1.reset_index(drop=True)

# 新一线城市国家队TOP20
top_national_new_tier1 = df_new_tier1.nlargest(20, '国家队Proportion')[
    ['李宁商场名称', '城市', '金标Proportion', '荣耀Proportion', '国家队Proportion', '其他Proportion']
].copy()
top_national_new_tier1 = top_national_new_tier1.reset_index(drop=True)

# ==================== 6. 保存TOP20结果 ====================
output_path = '一线和新一线高潜力商业体TOP20.xlsx'
with pd.ExcelWriter(output_path) as writer:
    top_gold_tier1.to_excel(writer, sheet_name='一线城市_金标TOP20', index=False)
    top_gold_new_tier1.to_excel(writer, sheet_name='新一线城市_金标TOP20', index=False)
    top_glory_tier1.to_excel(writer, sheet_name='一线城市_荣耀TOP20', index=False)
    top_glory_new_tier1.to_excel(writer, sheet_name='新一线城市_荣耀TOP20', index=False)
    top_national_tier1.to_excel(writer, sheet_name='一线城市_国家队TOP20', index=False)
    top_national_new_tier1.to_excel(writer, sheet_name='新一线城市_国家队TOP20', index=False)
    
    # 城市汇总
    df_all_filtered = df_predict[df_predict['城市'].isin(tier1_cities + new_tier1_cities)].copy()
    city_summary = df_all_filtered.groupby('城市').agg({
        '金标Proportion': 'mean',
        '荣耀Proportion': 'mean',
        '国家队Proportion': 'mean',
        '李宁商场名称': 'count'
    }).rename(columns={'李宁商场名称': '商业体数量'}).round(4)
    city_summary.to_excel(writer, sheet_name='城市汇总')
